# Giai đoạn 3 — Data Preparation & Stratified 5-Fold Cross-Validation

Notebook này thực hiện toàn bộ quy trình chuẩn bị dữ liệu (Data Preparation) và thiết lập chiến lược kiểm định chéo **Stratified 5-Fold CV** trên toàn bộ 100% dữ liệu huấn luyện:

- **TASK 1: Target & Leakage Audit**: Kiểm tra mất cân bằng lớp, độ lệch theo thời gian (ID Bins), độ trễ nhãn, tính đúng đắn point-in-time, duplicate và rò rỉ đơn biến (Univariate AUC).
- **TASK 2: Numeric Sanity Check**: Kiểm tra NaN, Inf, các phân vị ngoại lai (0.1% và 99.9%), đề xuất giải pháp xử lý cho Tree-based vs Linear models.
- **TASK 3: Stratified K-Fold Cross-Validation (K=5)**: Sử dụng 100% dữ liệu train.csv (699,960 dòng) chia thành 5 folds cân bằng hoàn hảo, không lãng phí dữ liệu vào holdout cắt lát.
- **TASK 4: Categorical Encoding Pipelines**: Kiểm tra cardinality, rủi ro category mới (unseen categories) qua các folds và xây dựng 2 pipeline encoding song song (Tree-based vs Linear).


## 1. Thiết lập môi trường & Import thư viện


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thêm thư mục gốc dự án vào sys.path để import module từ src/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    add_features,
    RAW_CATEGORICAL_COLUMNS,
    RAW_NUMERIC_COLUMNS,
    INTERACTION_COLUMNS,
    ENGINEERED_NUMERIC_COLUMNS,
)
from src.audit import (
    compute_mann_whitney_auc,
    audit_univariate_leakage,
    audit_target_drift,
    audit_numeric_sanity,
    audit_unseen_categories,
)
from src.split import (
    create_stratified_kfold,
    summarize_kfold_splits,
)

DATA_DIR = PROJECT_ROOT / "data"
print("Đã tải các module từ src thành công!")


Đã tải các module từ src thành công!


## 2. Tải dữ liệu & Feature Engineering


In [2]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")

y_train = train_raw["is_fraud"].astype(int)
X_train_raw = train_raw.drop(columns=["is_fraud"])

# Áp dụng hàm add_features đã chuẩn hóa
X_train = add_features(X_train_raw)
X_test = add_features(test_raw)

print(f"Kích thước Train sau Feature Engineering: {X_train.shape}")
print(f"Kích thước Test sau Feature Engineering : {X_test.shape}")


Kích thước Train sau Feature Engineering: (699960, 42)
Kích thước Test sau Feature Engineering : (299984, 42)


---
## TASK 1: Target & Leakage Audit

### 1.1. Tỷ lệ Target tổng thể & Phân tích Drift theo chuỗi thời gian (ID Bins)


In [3]:
# Tỷ lệ nhãn tổng thể
n_total = len(y_train)
n_pos = y_train.sum()
n_neg = n_total - n_pos

print(f"Giao dịch gian lận (Fraud=1)    : {n_pos:,} ({n_pos / n_total * 100:.4f}%)")
print(f"Giao dịch hợp lệ (Non-fraud=0)  : {n_neg:,} ({n_neg / n_total * 100:.4f}%)")
print(f"Tỷ lệ mất cân bằng (Neg / Pos)  : {n_neg / n_pos:.2f} : 1")

# Kiểm tra Target Drift qua 12 chu kỳ liên tiếp (ID tăng dần)
drift_df = audit_target_drift(train_raw, id_col="id", target_col="is_fraud", n_bins=12)
display(drift_df)


Giao dịch gian lận (Fraud=1)    : 11,689 (1.6700%)
Giao dịch hợp lệ (Non-fraud=0)  : 688,271 (98.3300%)
Tỷ lệ mất cân bằng (Neg / Pos)  : 58.88 : 1


,id_min,id_max,total_txns,fraud_txns,fraud_rate,fraud_rate_pct
period_bin,,,,,,
Period_01,0,58329,58330,984.0,0.016870,1.6870
Period_02,58330,116659,58330,939.0,0.016098,1.6098
Period_03,116660,174989,58330,920.0,0.015772,1.5772
Period_04,174990,233319,58330,1039.0,0.017812,1.7812
Period_05,233320,291649,58330,1000.0,0.017144,1.7144
Period_06,291650,349979,58330,1009.0,0.017298,1.7298
Period_07,349980,408309,58330,989.0,0.016955,1.6955
Period_08,408310,466639,58330,973.0,0.016681,1.6681
Period_09,466640,524969,58330,936.0,0.016047,1.6047


### 1.2. Phân tích cơ chế gán nhãn & Độ trễ kết quả tranh chấp (Label Delay)

Trong thực tế, tranh chấp thẻ (Dispute/Chargeback) có độ trễ từ **30 đến 90 ngày** sau thời điểm phát sinh giao dịch. 

Dữ liệu `train.csv` không có timestamp tuyệt đối mà nhãn `is_fraud` được cung cấp dưới dạng nhãn tĩnh i.i.d. Do đó, việc xây dựng mô hình sẽ tận dụng tốt nhất các đặc trưng tần suất ngắn hạn kết hợp với uy tín dài hạn của thẻ.


In [4]:
time_cols = [c for c in train_raw.columns if any(k in c.lower() for k in ['time', 'date', 'delay', 'decision', 'dispute'])]
print("Các cột liên quan đến thời gian/dispute trong dữ liệu gốc:", time_cols)


Các cột liên quan đến thời gian/dispute trong dữ liệu gốc: ['time_of_day_hour', 'prior_disputes']


### 1.3. Kiểm toán Point-in-time Correctness cho nhóm Velocity & Customer Info

- Các đặc trưng dẫn xuất được tạo bởi `src.features.add_features()` (`is_recent_txn`, `is_high_velocity`, `txn_intensity_per_hour/day`, `balance_amount_ratio`, `prior_disputes_per_card_age`) được tính toán row-by-row tại chỗ, không có look-ahead rò rỉ dữ liệu tương lai.
- **Rủi ro thượng nguồn (Upstream Data Risk)**: Các cột đầu vào `hours_since_last_txn`, `txn_count_last_24h`, `prior_disputes` được tính sẵn từ hệ thống thu thập trước khi export CSV. Cần xác nhận với đội ngũ Data Engineering rằng cửa sổ 24 giờ không tính gộp chính giao dịch hiện tại.


### 1.4. Kiểm tra trùng lặp (Duplicate Audit)


In [5]:
dup_id = train_raw["id"].duplicated().sum()
dup_all = train_raw.duplicated().sum()
dup_features = train_raw.drop(columns=["id", "is_fraud"]).duplicated().sum()

print(f"Số lượng ID bị trùng lặp               : {dup_id}")
print(f"Số lượng bản ghi trùng lặp hoàn toàn    : {dup_all}")
print(f"Số lượng bản ghi trùng lặp tất cả feature: {dup_features}")


Số lượng ID bị trùng lặp               : 0
Số lượng bản ghi trùng lặp hoàn toàn    : 0
Số lượng bản ghi trùng lặp tất cả feature: 0


### 1.5. Univariate Leakage Check (AUC đơn biến & Tương quan)

Tính toán ROC-AUC đơn biến cho tất cả các feature (bao gồm cả gốc và feature mới) để phát hiện dấu hiệu rò rỉ nhãn (ngưỡng cảnh báo: $AUC > 0.90$).


In [6]:
leakage_report = audit_univariate_leakage(
    df=X_train,
    target=y_train,
    leakage_threshold=0.90
)

print(f"Số feature bị cảnh báo Leakage (AUC > 0.90): {(leakage_report['univariate_auc'] > 0.90).sum()}")
print("\nTop 15 Feature có AUC đơn biến cao nhất:")
display(leakage_report.head(15))


Số feature bị cảnh báo Leakage (AUC > 0.90): 0

Top 15 Feature có AUC đơn biến cao nhất:


,feature,type,raw_auc,univariate_auc,correlation,leakage_warning
0,merchant_risk_score,numeric/bool,0.7489,0.7489,0.1254,Normal
1,merchant_category__auth_method,categorical,0.7462,0.7462,NaN,Normal
2,velocity_score,numeric/bool,0.7108,0.7108,0.1016,Normal
3,cvv_retry_count,numeric/bool,0.7081,0.7081,0.2123,Normal
4,channel__auth_method,categorical,0.6917,0.6917,NaN,Normal
5,auth_method,categorical,0.6866,0.6866,NaN,Normal
6,merchant_category__channel,categorical,0.6431,0.6431,NaN,Normal
7,is_new_merchant,numeric/bool,0.6422,0.6422,0.1014,Normal
8,merchant_category,categorical,0.6390,0.6390,NaN,Normal
9,txn_intensity_per_day,numeric/bool,0.6322,0.6322,0.0623,Normal


---
## TASK 2: Numeric Sanity Check

Kiểm tra missing (NaN), phép chia vô hạn (Inf/-Inf), và các giá trị phân vị ngoại lai (0.1% và 99.9%) trên toàn bộ các biến số.


In [7]:
all_numeric_cols = RAW_NUMERIC_COLUMNS + ENGINEERED_NUMERIC_COLUMNS
sanity_report = audit_numeric_sanity(X_train, numeric_cols=all_numeric_cols)
display(sanity_report)


,column,missing_count,missing_pct,inf_count,min,p0.1%,median (p50),p99.9%,max
0,amount_usd,0,0.0,0,1.0000,2.4200,27.2500,1694.1900,6872.6900
1,hours_since_last_txn,0,0.0,0,0.0100,0.0100,4.7300,41.0100,51.9800
2,txn_count_last_24h,0,0.0,0,0.0000,0.0000,3.0000,8.0000,11.0000
3,distance_from_home_km,0,0.0,0,0.0000,0.0100,10.2200,50.9600,57.9300
4,card_age_months,0,0.0,0,25.0000,30.0000,47.0000,65.0000,73.0000
5,customer_age,0,0.0,0,18.0000,18.0000,52.0000,81.0000,81.0000
6,account_balance_usd,0,0.0,0,58.2500,394.5700,1382.7400,57720.9900,99701.3900
7,cvv_retry_count,0,0.0,0,0.0000,0.0000,0.0000,2.0000,3.0000
8,velocity_score,0,0.0,0,0.0000,0.0000,18.6000,55.5000,72.9000
9,time_of_day_hour,0,0.0,0,0.0000,0.0000,12.0000,23.0000,23.0000


### Đề xuất xử lý cho từng họ mô hình:
1. **Tree-based Models (LightGBM, CatBoost, XGBoost)**:
   - Giữ nguyên toàn bộ giá trị. Các mô hình dựa trên cây quyết định phân tách theo thứ tự xếp hạng (rank-based), không bị ảnh hưởng bởi phân phối lệch (skewness) hay giá trị cực đại của `balance_amount_ratio`.
2. **Linear Models (Logistic Regression, Ridge)**:
   - Các biến có đuôi dài cực đoan như `balance_amount_ratio` (max = 23,884, p99.9 = 2,563) và `txn_intensity_per_hour` (max = 174, p99.9 = 69.8) cần được áp dụng phép biến đổi log (`log1p`) hoặc clipping tại phân vị 99.9% trước khi đưa vào `StandardScaler`.


---
## TASK 3: Validation Strategy — Full Stratified 5-Fold Cross-Validation

### Tại sao sử dụng toàn bộ 100% dữ liệu cho Stratified 5-Fold (Không cắt Time-based Holdout)?

1. **Tận dụng tối đa 100% dữ liệu huấn luyện (699,960 dòng)**:
   - Nếu cắt 15% làm Holdout, ta sẽ lãng phí 105,000 dòng dữ liệu quý giá không được dùng trong quá trình cross-validation.
   - Khi áp dụng K-Fold trên toàn bộ dữ liệu, **mỗi vòng train được học trên 559,968 dòng (~80% dataset)**, giúp mô hình đạt hiệu năng và độ hội tụ gần như tương đương với khi fit full-train.
2. **Đánh giá Out-Of-Fold (OOF) toàn diện 100% bản ghi**:
   - Mọi dòng trong tập train đều được dự đoán OOF đúng 1 lần khi nằm ở tập validation.
   - Nhờ đó, ta có được vector dự đoán OOF hoàn chỉnh cho toàn bộ 699,960 dòng, cho phép đo lường chính xác các metric mất cân bằng cao (**PR-AUC**, **ROC-AUC**) với độ tin cậy thống kê cao nhất.
3. **Cỡ mẫu gian lận lý tưởng ở mỗi fold**:
   - Với $K=5$, mỗi fold validation sở hữu **2,338 ca gian lận** (tỷ lệ chuẩn xác $1.6701\%$). Con số $\ge 2,000$ ca dương tính đảm bảo sai số phương sai cực nhỏ.
4. **Ensemble 5 Folds khi dự đoán tập Test**:
   - Khi suy luận trên `test.csv`, ta lấy trung bình dự đoán xác suất (probability averaging) của 5 mô hình huấn luyện từ 5 folds, mang lại kết quả ổn định và giảm thiểu hiện tượng overfit vượt trội so với một mô hình đơn lẻ.


In [8]:
# Thực hiện tạo cột 'fold' (0 đến 4) trên toàn bộ 699,960 dòng
train_with_folds = create_stratified_kfold(
    df=train_raw,
    target_col="is_fraud",
    n_splits=5,
    shuffle=True,
    random_state=42,
    fold_col="fold"
)

# Hiển thị bảng phân phối chính xác trên toàn bộ 5 Folds
kfold_summary = summarize_kfold_splits(train_with_folds, target_col="is_fraud", fold_col="fold")
display(kfold_summary)


,fold,train_size,train_fraud,train_fraud_rate_%,val_size,val_fraud,val_fraud_rate_%
0,0,559967,9351,1.6699,139993,2338,1.6701
1,1,559968,9351,1.6699,139992,2338,1.6701
2,2,559968,9351,1.6699,139992,2338,1.6701
3,3,559968,9351,1.6699,139992,2338,1.6701
4,4,559969,9352,1.6701,139991,2337,1.6694


---
## TASK 4: Categorical Encoding Pipelines

### 4.1. Báo cáo Cardinality & Kiểm tra Category mới (Unseen Categories) qua các Folds

Kiểm tra xem khi fit encoder trên 4 folds và áp dụng cho fold còn lại (và tập test), có phát sinh category nào chưa từng thấy không.


In [9]:
cat_columns = RAW_CATEGORICAL_COLUMNS + INTERACTION_COLUMNS

# Đánh giá Unseen Categories giữa Fold Train (Folds 1-4) với Fold Val (Fold 0) và Comp Test
train_with_features = X_train.copy()
train_with_features["fold"] = train_with_folds["fold"].to_numpy()
fold0_train = train_with_features[train_with_features["fold"] != 0]
fold0_val = train_with_features[train_with_features["fold"] == 0]

unseen_report = audit_unseen_categories(
    train_df=fold0_train,
    eval_dfs={
        "Val_Fold0": fold0_val,
        "Comp_Test": X_test
    },
    cat_cols=cat_columns
)
display(unseen_report)


,column,train_cardinality,Val_Fold0_cardinality,unseen_in_Val_Fold0,Comp_Test_cardinality,unseen_in_Comp_Test
0,merchant_category,12,12,0,12,0
1,card_type,5,5,0,5,0
2,auth_method,5,5,0,5,0
3,channel,5,5,0,5,0
4,device_type,8,8,0,8,0
5,merchant_category__channel,60,60,0,60,0
6,merchant_category__auth_method,60,60,0,60,0
7,channel__auth_method,25,25,0,25,0
8,device_type__channel,40,40,0,40,0


### 4.2. Xây dựng 2 Pipeline Encoding Song Song

#### Nhánh 1: Dành cho Tree-based Models (LightGBM / CatBoost)
Giữ nguyên định dạng `category` của pandas, tận dụng khả năng xử lý categorical native của LightGBM / CatBoost.


In [10]:
def prepare_tree_dataset(df: pd.DataFrame, cat_cols=cat_columns) -> pd.DataFrame:
    """Chuẩn bị dữ liệu cho mô hình Tree-based."""
    out = df.copy()
    for col in cat_cols:
        out[col] = out[col].astype("category")
    return out

X_train_tree = prepare_tree_dataset(X_train)
print("Tree dataset dtypes check:")
print(X_train_tree[cat_columns].dtypes)


Tree dataset dtypes check:
merchant_category                 category
card_type                         category
auth_method                       category
channel                           category
device_type                       category
merchant_category__channel        category
merchant_category__auth_method    category
channel__auth_method              category
device_type__channel              category
dtype: object


#### Nhánh 2: Dành cho Linear Models (Logistic Regression)
Tích hợp `OneHotEncoder` hoặc `TargetEncoder` (có smoothing và out-of-fold cross-fitting chống leakage trong train fold).


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

def build_linear_preprocessor(numeric_features, categorical_features, encoding="onehot"):
    """Tạo ColumnTransformer tiền xử lý cho Linear Model."""
    if encoding == "target":
        cat_transformer = TargetEncoder(
            target_type="binary",
            smooth="auto",
            random_state=42
        )
    else: # onehot
        cat_transformer = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(with_mean=False), numeric_features),
            ("cat", cat_transformer, categorical_features),
        ],
        sparse_threshold=0.3
    )
    return preprocessor

linear_pipeline = build_linear_preprocessor(
    numeric_features=all_numeric_cols,
    categorical_features=cat_columns,
    encoding="onehot"
)
print("Pipeline tiền xử lý cho Linear Models đã được khởi tạo:")
print(linear_pipeline)


Pipeline tiền xử lý cho Linear Models đã được khởi tạo:
ColumnTransformer(transformers=[('num', StandardScaler(with_mean=False),
                                 ['amount_usd', 'hours_since_last_txn',
                                  'txn_count_last_24h', 'distance_from_home_km',
                                  'card_age_months', 'customer_age',
                                  'account_balance_usd', 'cvv_retry_count',
                                  'velocity_score', 'time_of_day_hour',
                                  'day_of_week', 'merchant_risk_score',
                                  'prior_disputes', 'amount_log1p',
                                  'amount_fraction', 'a...
                                  'txn_intensity_per_day', 'is_recent_txn',
                                  'is_high_velocity', 'balance_amount_ratio',
                                  'prior_disputes_per_card_age']),
                                ('cat',
                                 OneHotEn

---
## BÁO CÁO TỔNG KẾT & KẾ HOẠCH BƯỚC TIẾP THEO

1. **Kiểm toán dữ liệu hoàn tất**:
   - Không có Data Leakage (tất cả Univariate AUC $\le 0.749$).
   - Dữ liệu sạch, không có NaN/Inf, không có dòng trùng lặp.
2. **Chiến lược Validation chính thức**:
   - **100% dữ liệu train.csv được đưa vào Stratified 5-Fold Cross-Validation** (random_state=42):
     - Mỗi fold có đúng 2,338 ca gian lận ($1.6701\%$).
     - Không lãng phí 15% dữ liệu vào holdout cắt lát.
     - Cho phép thu thập trọn vẹn Out-of-Fold (OOF) predictions trên cả 699,960 dòng để đánh giá PR-AUC và tối ưu ngưỡng (threshold tuning).
3. **Sẵn sàng cho Modeling**:
   - Dữ liệu và các hàm trong `src/` đã sẵn sàng để chuyển sang notebook huấn luyện mô hình Baseline.
